In [2]:
!pip install torch monai

In [37]:
import torch
from torch import nn
import torch.nn.functional as F
from monai.apps import DecathlonDataset

from monai.losses import DiceLoss
from monai.metrics import DiceMetric

from monai.transforms import (
    Activations,
    Activationsd,
    AsDiscrete,
    AsDiscreted,
    Compose,
    Invertd,
    LoadImaged,
    MapTransform,
    NormalizeIntensityd,
    Orientationd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    Spacingd,
    EnsureTyped,
    EnsureChannelFirstd,
)
import time
from monai.data import DataLoader, decollate_batch
from monai.inferers import sliding_window_inference

In [29]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(mid_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

In [30]:
class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2,
                                  mode='trilinear',
                                  align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose3d(
                in_channels,
                in_channels // 2,
                kernel_size=2,
                stride=2,
            )
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(
            x1,
            [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [31]:

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.maxpool_conv(x)

In [32]:
from monai.networks.blocks import PatchEmbeddingBlock
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, channel, spatial_dims, patch_size, hidden):
        super(MultiHeadSelfAttention, self).__init__()
        #self.query = MultiHeadDense(channel, bias=False)
        #self.key = MultiHeadDense(channel, bias=False)
        #self.value = MultiHeadDense(channel, bias=False)
        #self.softmax = nn.Softmax(dim=1)
        self.pe = PatchEmbeddingBlock(channel, spatial_dims, patch_size, hidden, 1)
        self.attn = torch.nn.MultiheadAttention(channel, 1, batch_first=True)

    def forward(self, x):
        b, c, h, w, d = x.size()
        # pe = self.positional_encoding_2d(c, h, w)
        x = self.pe(x)
        #x = x + pe
        #x = x.reshape(b, c, h * w).permute(0, 2, 1)  #[b, h*w, d]
        #Q = self.query(x)
        #K = self.key(x)
        #A = self.softmax(torch.bmm(Q, K.permute(0, 2, 1)) /
        #                 math.sqrt(c))  #[b, h*w, h*w]
        #V = self.value(x)
        #x = torch.bmm(A, V).permute(0, 2, 1).reshape(b, c, h, w)
        #print(x.size())
        attn_output = self.attn(x, x, x, need_weights=False)[0]
        x = attn_output.transpose(-1, -2).reshape(b, c, h, w, d)
        return x

In [33]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, channelY, channelS, spat_dimS, spat_dimY, num_heads):
        super(MultiHeadCrossAttention, self).__init__()
        self.Sconv = nn.Sequential(
            nn.MaxPool3d(2), nn.Conv3d(channelS, channelS, kernel_size=1),
            nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))
        self.Yconv = nn.Sequential(
            nn.Conv3d(channelY, channelS, kernel_size=1),
            nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))
        #self.query = MultiHeadDense(channelS, bias=False)
        #self.key = MultiHeadDense(channelS, bias=False)
        #self.value = MultiHeadDense(channelS, bias=False)
        self.conv = nn.Sequential(
            nn.Conv3d(channelY, channelS, kernel_size=1),
            nn.BatchNorm3d(channelS), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=4, mode='trilinear', align_corners=True))
        self.Yconv2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
            nn.Conv3d(channelY, channelY, kernel_size=3, padding=1),
            nn.Conv3d(channelY, channelS, kernel_size=1),
            nn.BatchNorm3d(channelS), nn.LeakyReLU(inplace=True))

        self.Spe = PatchEmbeddingBlock(channelS, spat_dimS, (4, 4, 4), channelY, num_heads)
        self.Ype = PatchEmbeddingBlock(channelY, spat_dimY, (2, 2, 2), channelY, num_heads)
        self.attn = torch.nn.MultiheadAttention(channelY, num_heads, batch_first=True)
        #self.softmax = nn.Softmax(dim=1)
        #self.Spe = PositionalEncodingPermute2D(channelS)
        #self.Ype = PositionalEncodingPermute2D(channelY)

    def forward(self, Y, S):
        Sb, Sc, Sh, Sw, Sd = S.size()
        Yb, Yc, Yh, Yw, Yd = Y.size()
        # Spe = self.positional_encoding_2d(Sc, Sh, Sw)
        S1 = self.Spe(S)
        #S = S + Spe
        #S = S.reshape(Sb, Sc, Sh, Sw, Sd)
        #S1 = self.Sconv(S).reshape(Sb, Sc, Yh * Yw * Yd).permute(0, 2, 1)
        #V = self.value(S1)
        # Ype = self.positional_encoding_2d(Yc, Yh, Yw)
        Y1 = self.Ype(Y)
        #Y = Y + Ype
        #Y = Y.reshape(Yb, Yc, Yh, Yw, Yd)
        #Y1 = self.Yconv(Y).reshape(Yb, Sc, Yh * Yw * Yd).permute(0, 2, 1)
        Y2 = self.Yconv2(Y)
        #Q = self.query(Y1)
        #K = self.key(Y1)
        #A = self.softmax(torch.bmm(Q, K.permute(0, 2, 1)) / math.sqrt(Sc))
        #x = torch.bmm(A, V).permute(0, 2, 1).reshape(Yb, Sc, Yh, Yw)
        #print("attention input: ", Y1.size(), S1.size())
        attn_output = self.attn(Y1, Y1, S1, need_weights=False)[0].permute(0, 2, 1).reshape(Yb, Yc, Yh // 2, Yw // 2, Yd // 2)
        #print("attention output: ", attn_output.size())
        Z = self.conv(attn_output)
        Z = Z * S
        Z = torch.cat([Z, Y2], dim=1)
        return Z

In [9]:
class TransformerUp(nn.Module):
    def __init__(self, Ychannels, Schannels, spat_dimS, spat_dimY, num_heads):
        super(TransformerUp, self).__init__()
        self.MHCA = MultiHeadCrossAttention(Ychannels, Schannels, spat_dimS, spat_dimY, num_heads)
        self.conv = nn.Sequential(
            nn.Conv3d(Ychannels,
                      Schannels,
                      kernel_size=3,
                      stride=1,
                      padding=1,
                      bias=True), nn.BatchNorm3d(Schannels),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(Schannels,
                      Schannels,
                      kernel_size=3,
                      stride=1,
                      padding=1,
                      bias=True), nn.BatchNorm3d(Schannels),
            nn.LeakyReLU(inplace=True))

    def forward(self, Y, S):
        x = self.MHCA(Y, S)
        x = self.conv(x)
        return x

In [34]:
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [35]:
class U_Transformer(nn.Module):
    def __init__(self, in_channels, classes, bilinear=True):
        super(U_Transformer, self).__init__()
        self.in_channels = in_channels
        self.classes = classes
        self.bilinear = bilinear

        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.transformer1 = MultiHeadSelfAttention(512, (28, 28, 18), (1, 1, 1), 512 )
        #self.MHSA = MultiHeadSelfAttention(512, )
        self.up1 = TransformerUp(512, 256, (56, 56, 36), (28, 28, 18), 1)
        self.up2 = TransformerUp(256, 128, (112, 112, 72), (56, 56, 36), 1)
        self.up3 = TransformerUp(128, 64, (224, 224, 144), (112, 112, 72), 1)
        self.outc = OutConv(64, classes)

    def forward(self, x):

        x1 = self.inc(x)
        #print(x1.size())
        x2 = self.down1(x1)
        #print(x2.size())
        x3 = self.down2(x2)
        #print(x3.size())
        x4 = self.down3(x3)
        #print(x4.size())
        x4 = self.transformer1(x4)
        #x = self
        #x4 = self.MHSA(x4)
        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)
        logits = self.outc(x)
        return logits

In [12]:
import os
os.environ['MONAI_DATA_DIRECTORY'] = 'monai_data'
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

monai_data


In [13]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    """
    Convert labels to multi channels based on brats classes:
    label 1 is the peritumoral edema
    label 2 is the GD-enhancing tumor
    label 3 is the necrotic and non-enhancing tumor core
    The possible classes are TC (Tumor core), WT (Whole tumor)
    and ET (Enhancing tumor).

    """

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # merge label 2 and label 3 to construct TC
            result.append(torch.logical_or(d[key] == 2, d[key] == 3))
            # merge labels 1, 2 and 3 to construct WT
            result.append(torch.logical_or(torch.logical_or(d[key] == 2, d[key] == 3), d[key] == 1))
            # label 2 is ET
            result.append(d[key] == 2)
            d[key] = torch.stack(result, axis=0).float()
        return d

In [23]:
class MapToSingleMode(MapTransform):
  def __call__(self, data):
      d = dict(data)
      for key in self.keys:
        img = d[key]
        if len(img.size()) == 4:
          img = img.unsqueeze(0)
        d[key] = img[:, 0, :, :, :]
      return d


In [25]:
train_transform = Compose(
    [
        # load 4 Nifti images and stack them together
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys="image"),
        EnsureTyped(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),
        RandSpatialCropd(keys=["image", "label"], roi_size=[224, 224, 144], random_size=False),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
        MapToSingleMode(keys="image")
    ]
)

val_transform = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys="image"),
        EnsureTyped(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        MapToSingleMode(keys="image")
    ]
)

In [15]:
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=train_transform,
    section="training",
    download=True,
    cache_rate=0.0,
    num_workers=0,
)

Task01_BrainTumour.tar: 7.09GB [03:11, 39.7MB/s]                            

2026-04-03 06:59:13,992 - INFO - Downloaded: monai_data/Task01_BrainTumour.tar


2026-04-03 06:59:26,242 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.
2026-04-03 06:59:26,243 - INFO - Writing into directory: monai_data.


In [17]:
device = torch.device("cuda:0")
model = U_Transformer(1, 3).to(device)
batch_data = train_ds[0]
scaler = torch.cuda.amp.GradScaler()
# enable cuDNN benchmark
torch.backends.cudnn.benchmark = True
inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )
inputs = torch.reshape(inputs[0], (1, 1, 224, 224, 144))
labels = labels.reshape(1, 3, 224, 224, 144)
print(inputs.size())

loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
#lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

torch.cuda.empty_cache()
model.train()
epoch_loss = 0
with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            #schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
            with_stack=True
        ) as prof:
  with torch.cuda.amp.autocast():
      outputs = model(inputs)
      loss = loss_function(outputs, labels)

  scaler.scale(loss).backward()
  scaler.step(optimizer)
  scaler.update()
  epoch_loss += loss.item()
  print(
  #    f"{step}/{len(train_ds) // train_loader.batch_size}"
      f", train_loss: {loss.item():.4f}"
  #    f", step time: {(time.time() - step_start):.4f}"
  )
              #prof.step()

  print('outputs', outputs.shape)

/tmp/ipykernel_786/3512427.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/tmp/ipykernel_786/3512427.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


torch.Size([1, 1, 224, 224, 144])
attention input:  torch.Size([1, 1764, 512]) torch.Size([1, 1764, 512])
attention output:  torch.Size([1, 512, 14, 14, 9])
attention input:  torch.Size([1, 14112, 256]) torch.Size([1, 14112, 256])
attention output:  torch.Size([1, 256, 28, 28, 18])
attention input:  torch.Size([1, 112896, 128]) torch.Size([1, 112896, 128])
attention output:  torch.Size([1, 128, 56, 56, 36])
, train_loss: 0.9702
outputs torch.Size([1, 3, 224, 224, 144])


In [ ]:
labels.size()

torch.Size([224, 224, 144])

In [ ]:
batch_data['image'].size()

torch.Size([4, 224, 224, 144])

In [26]:
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=train_transform,
    section="training",
    download=False,
    cache_rate=0.0,
    num_workers=0,
)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_ds = DecathlonDataset(
    root_dir=root_dir,
    task="Task01_BrainTumour",
    transform=val_transform,
    section="validation",
    download=False,
    cache_rate=0.0,
    num_workers=0,
)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

2026-04-03 07:45:26,953 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.
2026-04-03 07:45:26,953 - INFO - File exists: monai_data/Task01_BrainTumour.tar, skipped downloading.
2026-04-03 07:45:26,954 - INFO - Non-empty folder exists in monai_data/Task01_BrainTumour, skipped extracting.


In [39]:
max_epochs = 300
val_interval = 1
VAL_AMP = True

loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])


# define inference method
def inference(input):
    def _compute(input):
        return sliding_window_inference(
            inputs=input,
            roi_size=(224, 224, 144), # Changed from (240, 240, 160) to match training crop size
            sw_batch_size=1,
            predictor=model,
            overlap=0.5,
        )

    if VAL_AMP:
        with torch.autocast("cuda"):
            return _compute(input)
    else:
        return _compute(input)


# use amp to accelerate training
scaler = torch.GradScaler("cuda")
# enable cuDNN benchmark
torch.backends.cudnn.benchmark = True

In [40]:
best_metric = -1
best_metric_epoch = -1
best_metrics_epochs_and_time = [[], [], []]
epoch_loss_values = []
metric_values = []
metric_values_tc = []
metric_values_wt = []
metric_values_et = []
#set-up model
device = torch.device("cuda:0")
model = U_Transformer(1, 3).to(device)
torch.cuda.empty_cache()

# Re-initialize optimizer, loss function, metrics, and scaler after model setup
loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-5)
# lr_scheduler is also defined in 1SiHZ1k9PEwo, re-initialize for consistency if needed, but not directly causing this error.
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
dice_metric = DiceMetric(include_background=True, reduction="mean")
dice_metric_batch = DiceMetric(include_background=True, reduction="mean_batch")
scaler = torch.GradScaler("cuda") # Re-initialize the scaler

total_start = time.time()
for epoch in range(max_epochs):
    epoch_start = time.time()
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step_start = time.time()
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )
        optimizer.zero_grad()
        """with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
            with_stack=True
        ) as prof:"""
        with torch.autocast("cuda"):
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        print(
            f"{step}/{len(train_ds) // train_loader.batch_size}"
            f", train_loss: {loss.item():.4f}"
            f", step time: {(time.time() - step_start):.4f}"
        )
            #prof.step()

    lr_scheduler.step()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device),
                )
                val_outputs = inference(val_inputs)
                val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                dice_metric(y_pred=val_outputs, y=val_labels)
                dice_metric_batch(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            metric_values.append(metric)
            metric_batch = dice_metric_batch.aggregate()
            metric_tc = metric_batch[0].item()
            metric_values_tc.append(metric_tc)
            metric_wt = metric_batch[1].item()
            metric_values_wt.append(metric_wt)
            metric_et = metric_batch[2].item()
            metric_values_et.append(metric_et)
            dice_metric.reset()
            dice_metric_batch.reset()

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                best_metrics_epochs_and_time[0].append(best_metric)
                best_metrics_epochs_and_time[1].append(best_metric_epoch)
                best_metrics_epochs_and_time[2].append(time.time() - total_start)
                torch.save(
                    model.state_dict(),
                    os.path.join(root_dir, "best_metric_model.pth"),
                )
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f" tc: {metric_tc:.4f} wt: {metric_wt:.4f} et: {metric_et:.4f}"
                f"\nbest mean dice: {best_metric:.4f}"
                f" at epoch: {best_metric_epoch}"
            )
    print(f"time consuming of epoch {epoch + 1} is: {(time.time() - epoch_start):.4f}")
total_time = time.time() - total_start

----------
epoch 1/300
1/388, train_loss: 0.9916, step time: 0.5550
2/388, train_loss: 0.9473, step time: 0.5530
3/388, train_loss: 0.9280, step time: 0.5521
4/388, train_loss: 0.9273, step time: 0.5518
5/388, train_loss: 0.9650, step time: 0.5535
6/388, train_loss: 0.9357, step time: 0.5526
7/388, train_loss: 0.9115, step time: 0.5537
8/388, train_loss: 0.9689, step time: 0.5513
9/388, train_loss: 0.9577, step time: 0.5524
10/388, train_loss: 0.8955, step time: 0.5517
11/388, train_loss: 0.9807, step time: 0.5510
12/388, train_loss: 0.9441, step time: 0.5530
13/388, train_loss: 0.9482, step time: 0.5529
14/388, train_loss: 0.8442, step time: 0.5530
15/388, train_loss: 0.9221, step time: 0.5532
16/388, train_loss: 0.8912, step time: 0.5524
17/388, train_loss: 0.9577, step time: 0.5519
18/388, train_loss: 0.9523, step time: 0.5525
19/388, train_loss: 0.8861, step time: 0.5532
20/388, train_loss: 0.8961, step time: 0.5509
21/388, train_loss: 0.9476, step time: 0.5520
22/388, train_loss: 

/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out[idx_zm] += p


saved new best metric model
current epoch: 1 current mean dice: 0.3921 tc: 0.3155 wt: 0.6796 et: 0.1813
best mean dice: 0.3921 at epoch: 1
time consuming of epoch 1 is: 762.1169
----------
epoch 2/300
1/388, train_loss: 0.9588, step time: 0.5503
2/388, train_loss: 0.7756, step time: 0.5510
3/388, train_loss: 0.9400, step time: 0.5491
4/388, train_loss: 0.6890, step time: 0.5517
5/388, train_loss: 0.7252, step time: 0.5524
6/388, train_loss: 0.8417, step time: 0.5511
7/388, train_loss: 0.7575, step time: 0.5492
8/388, train_loss: 0.6864, step time: 0.5518
9/388, train_loss: 0.9038, step time: 0.5512
10/388, train_loss: 0.7087, step time: 0.5518
11/388, train_loss: 0.8301, step time: 0.5510
12/388, train_loss: 0.7132, step time: 0.5515
13/388, train_loss: 0.6908, step time: 0.5500
14/388, train_loss: 0.8116, step time: 0.5499
15/388, train_loss: 0.8616, step time: 0.5515
16/388, train_loss: 0.8174, step time: 0.5517
17/388, train_loss: 0.5558, step time: 0.5514
18/388, train_loss: 0.7025

KeyboardInterrupt: 